##  Data Ingestion

In [1]:
from langchain_core.documents import Document
from openai import embeddings

/Users/rhalder/anaconda3/envs/RAG/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [8]:
doc = Document(
    page_content = "Here is a tutorial of RAG.",
    metadata = {"page":1}
)

In [9]:
doc

Document(page_content='Here is a tutorial of RAG.', metadata={'page': 1})

In [10]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [12]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [20]:
# TextLoader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")


In [16]:
loader

In [19]:
document = loader.load()
print(document)

[Document(page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.', metadata={'source': '../data/text_files/python_intro.txt'})]


In [25]:
# Directory loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)

In [26]:
document=dir_loader.load()
print(document)

100%|██████████| 2/2 [00:00<00:00, 1476.09it/s]

[Document(page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.', metadata={'source': '../data/text_files/python_intro.txt'}), Document(page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning:

In [29]:
# Directory loader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyPDFLoader,
    show_progress=True
)

In [34]:
pdf_document = dir_loader.load()

100%|██████████| 2/2 [00:00<00:00,  3.68it/s]


In [37]:
pdf_document[0].schema()

{'title': 'Document',
 'description': 'Class for storing a piece of text and associated metadata.',
 'type': 'object',
 'properties': {'page_content': {'title': 'Page Content', 'type': 'string'},
  'metadata': {'title': 'Metadata', 'type': 'object'},
  'type': {'title': 'Type',
   'default': 'Document',
   'enum': ['Document'],
   'type': 'string'}},
 'required': ['page_content']}

## Chunking

In [39]:
## Text splitting into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks of given size."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", "", " "]
    )

    split_doc = text_splitter.split_documents(documents)
    print("Split documents done!")

    if split_doc:
        print("Example of chunk!")
        print(f"Content: {split_doc[0].page_content[0:200]}..")
        print(f"Metadata: {split_doc[0].metadata}..")

    return split_doc



In [40]:
chunks=split_documents(pdf_document)
chunks

Split documents done!
Example of chunk!
Content: Transformers are RNNs:
Fast Autoregressive Transformers with Linear Attention
Angelos Katharopoulos 1 2 Apoorv Vyas 1 2 Nikolaos Pappas 3 Franc ¸ois Fleuret2 4 *
Abstract
Transformers achieve remarkab..
Metadata: {'source': '../data/pdf/attention.pdf', 'page': 0}..


[Document(page_content='Transformers are RNNs:\nFast Autoregressive Transformers with Linear Attention\nAngelos Katharopoulos 1 2 Apoorv Vyas 1 2 Nikolaos Pappas 3 Franc ¸ois Fleuret2 4 *\nAbstract\nTransformers achieve remarkable performance in\nseveral tasks but due to their quadratic complex-\nity, with respect to the input’s length, they are\nprohibitively slow for very long sequences. To ad-\ndress this limitation, we express the self-attention\nas a linear dot-product of kernel feature maps and\nmake use of the associativity property of matrix\nproducts to reduce the complexity fromO\n(\nN2)\ntoO (N), where N is the sequence length. We\nshow that this formulation permits an iterative\nimplementation that dramatically accelerates au-\ntoregressive transformers and reveals their rela-\ntionship to recurrent neural networks. Our lin-\near transformers achieve similar performance to\nvanilla transformers and they are up to 4000x\nfaster on autoregressive prediction of very long\nsequ

## Embedding and VectorStore DB

In [41]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading {self.model_name} model...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded! with model embeddings dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Failed to load {self.model_name} model: {e}")
            raise e
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Embeddings shape: {embeddings.shape}")
        return embeddings



In [55]:
### Initialize Embedding Manager
embedding_manager = EmbeddingManager()
# embedding_manager.generate_embeddings(sample_texts["../data/text_files/python_intro.txt"])



Loading all-MiniLM-L6-v2 model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9609.69it/s]
/var/folders/_4/mt2jwyyd3ls_p621k2sz8vhr0000gn/T/ipykernel_65498/787336789.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded! with model embeddings dimension: {self.model.get_sentence_embedding_dimension()}")


Model loaded! with model embeddings dimension: 384


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]

Embeddings shape: (384,)


array([-5.52211553e-02,  1.13732042e-02, -1.35107692e-02,  4.44357730e-02,
       -2.84066647e-02, -1.15500830e-01, -1.46656781e-02,  5.57077341e-02,
       -9.28853974e-02, -1.94044616e-02, -8.43826756e-02,  8.78921971e-02,
        8.02288353e-02,  3.33797038e-02,  7.12218136e-02, -2.74255946e-02,
       -1.91296171e-02, -2.64329631e-02,  3.34410518e-02, -9.20909867e-02,
       -5.34203984e-02,  7.42993951e-02, -6.15640543e-03, -3.06963306e-02,
       -1.58410391e-03, -2.92596109e-02,  1.88450434e-03, -6.70600682e-03,
        1.28711201e-02,  2.23941952e-02, -3.32809277e-02,  3.29755992e-02,
        2.96935253e-02, -2.11348361e-03,  8.37673433e-03,  2.39775646e-02,
        8.37997868e-05, -7.69552439e-02, -6.63860962e-02,  4.47619818e-02,
       -6.09366447e-02,  4.04922962e-02, -7.71294162e-02, -5.56969196e-02,
       -6.71717599e-02,  5.69408908e-02, -5.85117098e-03, -6.25167415e-03,
       -5.51510649e-03, -1.16926879e-02, -7.67502263e-02,  6.46434398e-03,
        1.39465043e-02, -

In [44]:
## Vector store
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [57]:
## Convert text to embeddings
texts = [doc.page_content for doc in chunks]

## Genrate the embeddings
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Loading all-MiniLM-L6-v2 model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6508.68it/s]
/var/folders/_4/mt2jwyyd3ls_p621k2sz8vhr0000gn/T/ipykernel_65498/787336789.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded! with model embeddings dimension: {self.model.get_sentence_embedding_dimension()}")


Model loaded! with model embeddings dimension: 384


Batches: 100%|██████████| 4/4 [00:00<00:00,  5.02it/s]


Embeddings shape: (116, 384)
Adding 116 documents to vector store...
Successfully added 116 documents to vector store
Total documents in collection: 116


## Retriever Pipeline from Vectorstore

In [58]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [60]:
rag_retriever.retrieve("What is Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention")

Retrieving documents for query: 'What is Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]

Embeddings shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d8e4b959_0',
  'content': 'Transformers are RNNs:\nFast Autoregressive Transformers with Linear Attention\nAngelos Katharopoulos 1 2 Apoorv Vyas 1 2 Nikolaos Pappas 3 Franc ¸ois Fleuret2 4 *\nAbstract\nTransformers achieve remarkable performance in\nseveral tasks but due to their quadratic complex-\nity, with respect to the input’s length, they are\nprohibitively slow for very long sequences. To ad-\ndress this limitation, we express the self-attention\nas a linear dot-product of kernel feature maps and\nmake use of the associativity property of matrix\nproducts to reduce the complexity fromO\n(\nN2)\ntoO (N), where N is the sequence length. We\nshow that this formulation permits an iterative\nimplementation that dramatically accelerates au-\ntoregressive transformers and reveals their rela-\ntionship to recurrent neural networks. Our lin-\near transformers achieve similar performance to\nvanilla transformers and they are up to 4000x\nfaster on autoregressive prediction of

## Integration Vectordb Context pipeline With LLM output

In [61]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


ImportError: cannot import name 'ensure_id' from 'langchain_core.utils.utils' (/Users/rhalder/anaconda3/envs/RAG/lib/python3.12/site-packages/langchain_core/utils/utils.py)